# EEGNet Pure Deep Learning From EDF (LH vs RH, Local)

This notebook follows the original EEGNet-style EDF pipeline, but runs entirely on local paths and only uses scenario 1 (left hand) and scenario 2 (right hand).

Pipeline summary:
1. Load EDF directly from local dataset folders.
2. Apply bandpass filtering and window segmentation.
3. Build train/validation/test splits by subject.
4. Train EEGNet on windows.
5. Save metrics, plots, and trained weights.

In [1]:
import os
import json
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import butter, filtfilt

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, cohen_kappa_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Local configuration
BASE_DIR = Path('/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook')
DATASET_PATH = Path('/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/dataset/EEGET-ALS Dataset/dataset')
RESULTS_DIR = BASE_DIR / 'results_local_cpu_gpu' / 'deep_learning_edf_lh_rh'
MODELS_DIR = BASE_DIR / 'results_local_cpu_gpu' / 'saved_models'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = 'smoke'  # switch to 'full' for full training
RANDOM_STATE = 42

# Signal settings aligned to original notebook
TARGET_SFREQ = 128
WINDOW_DURATION = 2
WINDOW_SAMPLES = TARGET_SFREQ * WINDOW_DURATION
WINDOW_OVERLAP = 0.5
LOW_FREQ = 8.0
HIGH_FREQ = 30.0
N_CHANNELS = 32

# Only left hand and right hand
SCENARIO_TO_LABEL = {1: 0, 2: 1}
LABEL_TO_NAME = {0: 'LH', 1: 'RH'}

# Runtime controls
SMOKE_MAX_SUBJECTS = 16
SMOKE_MAX_WINDOWS_PER_SCENARIO = 24
BATCH_SIZE = 32 if RUN_MODE == 'smoke' else 128
MAX_EPOCHS = 4 if RUN_MODE == 'smoke' else 50
PATIENCE = 2 if RUN_MODE == 'smoke' else 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_STATE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

print('Data path:', DATASET_PATH)
print('Results path:', RESULTS_DIR)
print('Run mode:', RUN_MODE)

Device: cuda
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU
Data path: /home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/dataset/EEGET-ALS Dataset/dataset
Results path: /home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook/results_local_cpu_gpu/deep_learning_edf_lh_rh
Run mode: smoke


In [2]:
# Utilities copied in spirit from the original pipeline
def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return b, a

def apply_bandpass_filter(data, lowcut, highcut, fs, order=4):
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    return filtfilt(b, a, data, axis=-1)

def z_score_normalize(window):
    mu = np.mean(window, axis=-1, keepdims=True)
    sigma = np.std(window, axis=-1, keepdims=True)
    sigma[sigma == 0] = 1.0
    return (window - mu) / sigma

def segment_data(data, window_samples, overlap=0.5):
    n_channels, n_samples = data.shape
    step = int(window_samples * (1 - overlap))
    if step <= 0 or n_samples < window_samples:
        return np.empty((0, n_channels, window_samples), dtype=np.float32)

    segments = []
    for start in range(0, n_samples - window_samples + 1, step):
        seg = data[:, start:start + window_samples]
        if seg.shape[1] == window_samples:
            segments.append(seg)

    if not segments:
        return np.empty((0, n_channels, window_samples), dtype=np.float32)
    return np.asarray(segments, dtype=np.float32)

def load_edf_data(edf_path, target_sfreq=128, n_channels=32):
    try:
        raw = mne.io.read_raw_edf(str(edf_path), preload=True, verbose=False)
        if int(round(raw.info['sfreq'])) != int(target_sfreq):
            raw.resample(target_sfreq, npad='auto')
        data = raw.get_data()
        if data.shape[0] < n_channels:
            return None
        return data[:n_channels, :]
    except Exception:
        return None

def parse_scenario_id(scenario_folder):
    scenario_json = scenario_folder / 'scenario.json'
    if scenario_json.exists():
        try:
            with open(scenario_json, 'r', encoding='utf-8') as f:
                data = json.load(f)
            sid = int(data.get('scenarioId', -1))
            if sid > 0:
                return sid
        except Exception:
            pass

    name = scenario_folder.name.lower().replace('scenario', '').strip()
    try:
        return int(name)
    except Exception:
        return -1

def collect_subject_windows(subject_dir, smoke=False):
    X_parts = []
    y_parts = []

    time_folders = sorted(subject_dir.glob('time*'))
    for time_folder in time_folders:
        for scenario_folder in sorted(time_folder.glob('scenario*')):
            sid = parse_scenario_id(scenario_folder)
            if sid not in SCENARIO_TO_LABEL:
                continue

            edf_path = scenario_folder / 'EEG.edf'
            if not edf_path.exists():
                continue

            data = load_edf_data(edf_path, target_sfreq=TARGET_SFREQ, n_channels=N_CHANNELS)
            if data is None:
                continue

            filtered = apply_bandpass_filter(data, LOW_FREQ, HIGH_FREQ, TARGET_SFREQ)
            windows = segment_data(filtered, WINDOW_SAMPLES, overlap=WINDOW_OVERLAP)
            if len(windows) == 0:
                continue

            if smoke:
                windows = windows[:SMOKE_MAX_WINDOWS_PER_SCENARIO]

            windows = np.asarray([z_score_normalize(w) for w in windows], dtype=np.float32)
            labels = np.full((len(windows),), SCENARIO_TO_LABEL[sid], dtype=np.int64)

            X_parts.append(windows)
            y_parts.append(labels)

    if not X_parts:
        return None, None

    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    return X, y

In [3]:
# Load healthy subjects only and keep scenarios 1/2
subject_dirs = sorted([p for p in DATASET_PATH.glob('id*') if p.is_dir()])
if RUN_MODE == 'smoke':
    subject_dirs = subject_dirs[:SMOKE_MAX_SUBJECTS]

print(f'Healthy subjects to process: {len(subject_dirs)}')

X_all, y_all, s_all = [], [], []
per_subject_counts = {}

for subject_dir in subject_dirs:
    subject_id = subject_dir.name
    X_subj, y_subj = collect_subject_windows(subject_dir, smoke=(RUN_MODE == 'smoke'))
    if X_subj is None:
        continue

    X_all.append(X_subj)
    y_all.append(y_subj)
    s_all.append(np.array([subject_id] * len(y_subj)))
    per_subject_counts[subject_id] = int(len(y_subj))

if not X_all:
    raise RuntimeError('No EEG windows collected. Check dataset path and file integrity.')

X = np.concatenate(X_all, axis=0).astype(np.float32)    # [N, C, T]
y = np.concatenate(y_all, axis=0).astype(np.int64)      # [N]
subjects = np.concatenate(s_all, axis=0).astype(str)    # [N]

print(f'Collected windows: {X.shape}')
print(f'Collected labels: {y.shape}')
print('Label counts:', {LABEL_TO_NAME[k]: int((y == k).sum()) for k in sorted(np.unique(y))})
print(f'Unique subjects with data: {len(np.unique(subjects))}')

class_df = pd.DataFrame({
    'label': [LABEL_TO_NAME[i] for i in sorted(np.unique(y))],
    'count': [int((y == i).sum()) for i in sorted(np.unique(y))],
})
plt.figure(figsize=(6, 4))
sns.barplot(data=class_df, x='label', y='count', palette='Set2')
plt.title('Class Distribution (LH vs RH)')
for i, v in enumerate(class_df['count'].values):
    plt.text(i, v, str(int(v)), ha='center', va='bottom')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'class_distribution_lh_rh.png', dpi=180)
plt.close()

Healthy subjects to process: 16
Collected windows: (768, 32, 256)
Collected labels: (768,)
Label counts: {'LH': 384, 'RH': 384}
Unique subjects with data: 16


In [4]:
# Subject-wise split to reduce leakage
unique_subjects = np.array(sorted(np.unique(subjects)))
if len(unique_subjects) < 4:
    raise RuntimeError('Need at least 4 subjects for train/val/test split.')

train_subjects, test_subjects = train_test_split(
    unique_subjects, test_size=0.2, random_state=RANDOM_STATE
)
train_subjects, val_subjects = train_test_split(
    train_subjects, test_size=0.2, random_state=RANDOM_STATE
)

train_mask = np.isin(subjects, train_subjects)
val_mask = np.isin(subjects, val_subjects)
test_mask = np.isin(subjects, test_subjects)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print('Train:', X_train.shape, y_train.shape, 'subjects:', len(np.unique(subjects[train_mask])))
print('Val  :', X_val.shape, y_val.shape, 'subjects:', len(np.unique(subjects[val_mask])))
print('Test :', X_test.shape, y_test.shape, 'subjects:', len(np.unique(subjects[test_mask])))

def class_ratio(name, labels):
    vals = {LABEL_TO_NAME[k]: int((labels == k).sum()) for k in sorted(np.unique(y))}
    print(name, vals)

class_ratio('Train class counts ->', y_train)
class_ratio('Val class counts   ->', y_val)
class_ratio('Test class counts  ->', y_test)

Train: (432, 32, 256) (432,) subjects: 9
Val  : (144, 32, 256) (144,) subjects: 3
Test : (192, 32, 256) (192,) subjects: 4
Train class counts -> {'LH': 216, 'RH': 216}
Val class counts   -> {'LH': 72, 'RH': 72}
Test class counts  -> {'LH': 96, 'RH': 96}


In [5]:
class EEGWindowDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X = torch.tensor(X_data, dtype=torch.float32)
        self.y = torch.tensor(y_data, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(EEGWindowDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(EEGWindowDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(EEGWindowDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('DataLoaders ready:', len(train_loader), len(val_loader), len(test_loader))

DataLoaders ready: 14 5 6


In [6]:
# EEGNet architecture aligned with original style (Conv2D over [channels x time])
class EEGNet(nn.Module):
    def __init__(self, n_classes=2, chans=32, samples=256, dropout=0.5, F1=8, D=2, F2=16, kern_length=64):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kern_length), padding=(0, kern_length // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, kernel_size=(chans, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            x = torch.zeros(1, 1, chans, samples)
            x = self.block1(x)
            x = self.block2(x)
            flatten_dim = x.flatten(1).shape[1]

        self.classifier = nn.Linear(flatten_dim, n_classes)

    def forward(self, x):
        # x: [B, C, T] -> [B, 1, C, T]
        x = x.unsqueeze(1)
        x = self.block1(x)
        x = self.block2(x)
        x = x.flatten(1)
        return self.classifier(x)

model = EEGNet(n_classes=2, chans=N_CHANNELS, samples=WINDOW_SAMPLES).to(device)

class_counts = np.bincount(y_train, minlength=2)
class_weights = torch.tensor([1.0 / max(class_counts[0], 1), 1.0 / max(class_counts[1], 1)], dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-5)

print(model)
print('Class weights:', class_weights.detach().cpu().numpy())

EEGNet(
  (block1): Sequential(
    (0): Conv2d(1, 8, kernel_size=(1, 64), stride=(1, 1), padding=(0, 32), bias=False)
    (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Conv2d(8, 16, kernel_size=(32, 1), stride=(1, 1), groups=8, bias=False)
    (3): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): ELU(alpha=1.0)
    (5): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
    (6): Dropout(p=0.5, inplace=False)
  )
  (block2): Sequential(
    (0): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=(0, 8), groups=16, bias=False)
    (1): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ELU(alpha=1.0)
    (4): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
    (5): Dropout(p=0.5, inplace=False)
  )
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)
Class w

In [7]:
def run_epoch(loader, training=False):
    if training:
        model.train()
    else:
        model.eval()

    all_true, all_pred, all_prob = [], [], []
    total_loss = 0.0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = criterion(logits, yb)
            probs = torch.softmax(logits, dim=1)[:, 1]
            pred = torch.argmax(logits, dim=1)

            if training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * len(yb)
        all_true.append(yb.detach().cpu().numpy())
        all_pred.append(pred.detach().cpu().numpy())
        all_prob.append(probs.detach().cpu().numpy())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    y_prob = np.concatenate(all_prob)

    out = {
        'loss': total_loss / max(len(y_true), 1),
        'acc': accuracy_score(y_true, y_pred),
        'bac': balanced_accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }
    return out

history = []
best_state = None
best_val_bac = -np.inf
best_epoch = 0

for epoch in range(1, MAX_EPOCHS + 1):
    tr = run_epoch(train_loader, training=True)
    va = run_epoch(val_loader, training=False)
    scheduler.step()

    history.append({
        'epoch': epoch,
        'train_loss': tr['loss'],
        'val_loss': va['loss'],
        'train_acc': tr['acc'],
        'val_acc': va['acc'],
        'val_bac': va['bac'],
        'val_auc': va['auc'],
    })

    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
        f"train_loss={tr['loss']:.4f}, val_loss={va['loss']:.4f}, "
        f"train_acc={tr['acc']:.4f}, val_acc={va['acc']:.4f}, val_bac={va['bac']:.4f}, val_auc={va['auc']:.4f}"
    )

    if va['bac'] > best_val_bac:
        best_val_bac = va['bac']
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch - best_epoch >= PATIENCE:
        print('Early stopping triggered.')
        break

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
history_df.to_csv(RESULTS_DIR / 'training_history_edf_lh_rh.csv', index=False)

plt.figure(figsize=(10, 4))
plt.plot(history_df['epoch'], history_df['train_loss'], label='Train Loss')
plt.plot(history_df['epoch'], history_df['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curves - Loss')
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curve_loss_edf_lh_rh.png', dpi=180)
plt.close()

plt.figure(figsize=(10, 4))
plt.plot(history_df['epoch'], history_df['train_acc'], label='Train Accuracy')
plt.plot(history_df['epoch'], history_df['val_acc'], label='Val Accuracy')
plt.plot(history_df['epoch'], history_df['val_bac'], label='Val BAC')
if history_df['val_auc'].notna().any():
    plt.plot(history_df['epoch'], history_df['val_auc'], label='Val AUC')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Training Curves - Scores')
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curve_scores_edf_lh_rh.png', dpi=180)
plt.close()

Epoch 01/4 | train_loss=0.7024, val_loss=0.6932, train_acc=0.4815, val_acc=0.4931, val_bac=0.4931, val_auc=0.5064
Epoch 02/4 | train_loss=0.6990, val_loss=0.6934, train_acc=0.5208, val_acc=0.4861, val_bac=0.4861, val_auc=0.5008
Epoch 03/4 | train_loss=0.7114, val_loss=0.6939, train_acc=0.4722, val_acc=0.4653, val_bac=0.4653, val_auc=0.4946
Early stopping triggered.


In [8]:
# Final evaluation on held-out test subjects
te = run_epoch(test_loader, training=False)

test_metrics = {
    'accuracy': te['acc'],
    'balanced_accuracy': te['bac'],
    'f1': te['f1'],
    'roc_auc': te['auc'],
    'cohen_kappa': cohen_kappa_score(te['y_true'], te['y_pred']),
    'best_val_bac': best_val_bac,
    'best_epoch': best_epoch,
    'run_mode': RUN_MODE,
    'device': str(device),
    'n_train': int(len(y_train)),
    'n_val': int(len(y_val)),
    'n_test': int(len(y_test)),
}

print('Test metrics:')
for k, v in test_metrics.items():
    print(f'- {k}: {v}')

print('Classification report:')
print(classification_report(te['y_true'], te['y_pred'], target_names=['LH', 'RH'], zero_division=0))

pd.DataFrame([test_metrics]).to_csv(RESULTS_DIR / 'test_metrics_edf_lh_rh.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm = confusion_matrix(te['y_true'], te['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['LH', 'RH']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix (Counts)')
cmn = confusion_matrix(te['y_true'], te['y_pred'], normalize='true')
ConfusionMatrixDisplay(cmn, display_labels=['LH', 'RH']).plot(ax=axes[1], colorbar=False)
axes[1].set_title('Confusion Matrix (Normalized)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix_edf_lh_rh.png', dpi=180)
plt.close()

fpr, tpr, _ = roc_curve(te['y_true'], te['y_prob'])
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"EEGNet (AUC={te['auc']:.3f})")
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - EDF LH vs RH')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'roc_curve_edf_lh_rh.png', dpi=180)
plt.close()

model_path = MODELS_DIR / 'eegnet_edf_lh_rh_best.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'n_channels': N_CHANNELS,
        'window_samples': WINDOW_SAMPLES,
        'target_sfreq': TARGET_SFREQ,
        'low_freq': LOW_FREQ,
        'high_freq': HIGH_FREQ,
        'scenario_to_label': SCENARIO_TO_LABEL,
        'label_to_name': LABEL_TO_NAME,
        'run_mode': RUN_MODE,
    },
    'metrics': test_metrics,
}, model_path)

subject_count_df = pd.DataFrame([
    {'subject_id': s, 'n_windows': int(c)} for s, c in sorted(per_subject_counts.items())
])
subject_count_df.to_csv(RESULTS_DIR / 'subject_window_counts.csv', index=False)

print('Saved files in results folder:')
for p in sorted(RESULTS_DIR.glob('*')):
    if p.is_file():
        print(f'- {p.name} ({p.stat().st_size / 1024:.1f} KB)')

print('Saved model files:')
for p in sorted(MODELS_DIR.glob('eegnet_edf_lh_rh*')):
    if p.is_file():
        print(f'- {p.name} ({p.stat().st_size / 1024:.1f} KB)')

Test metrics:
- accuracy: 0.4895833333333333
- balanced_accuracy: 0.4895833333333333
- f1: 0.057692307692307696
- roc_auc: 0.4763997395833333
- cohen_kappa: -0.02083333333333326
- best_val_bac: 0.4930555555555556
- best_epoch: 1
- run_mode: smoke
- device: cuda
- n_train: 432
- n_val: 144
- n_test: 192
Classification report:
              precision    recall  f1-score   support

          LH       0.49      0.95      0.65        96
          RH       0.38      0.03      0.06        96

    accuracy                           0.49       192
   macro avg       0.43      0.49      0.35       192
weighted avg       0.43      0.49      0.35       192

Saved files in results folder:
- class_distribution_lh_rh.png (33.4 KB)
- confusion_matrix_edf_lh_rh.png (46.2 KB)
- roc_curve_edf_lh_rh.png (69.7 KB)
- subject_window_counts.csv (0.2 KB)
- test_metrics_edf_lh_rh.csv (0.2 KB)
- training_curve_loss_edf_lh_rh.png (74.1 KB)
- training_curve_scores_edf_lh_rh.png (106.3 KB)
- training_history_edf_lh